# PfMYST × dihydroartemisinin single-cell analysis

**Question.** Which gene-level responses to DHA are directionally similar across time and glucosamine strata in the public PfMYST experiment?

**Sources.** [GEO GSE246115](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE246115); [primary study](https://pmc.ncbi.nlm.nih.gov/articles/PMC12394585/). The matrix is a ten-group subset with one GSM/library per group. G1 denotes glucosamine exposure of the Pfmyst-ty1-glmS line; G0 lacks glucosamine. Group names use DHA0/DHA1. We compare treatment with time-matched controls within each G stratum.

**Method.** Download and checksum the deposited raw-count matrix; stream gene rows to sum counts by GSM; compute group pseudobulk CPM; summarize log2(CPM+1) treated minus control. Filter genes with control CPM ≥1 for each contrast, and use the 3,069-gene intersection for descriptive Spearman rank comparisons. Cell-level summaries are quality context only. No differential-expression p-values or gene-set enrichment are calculated.

In [1]:
from pathlib import Path
import importlib.util
ROOT = Path.cwd()
script = ROOT / 'research/transcriptomics/gse246115/analysis.py'
spec = importlib.util.spec_from_file_location('pfmyst_analysis', script)
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
summary = analysis.run(ROOT)
summary

{'accession': 'GSE246115',
 'matrix_sha256': '80c558a843104e98dd57753a1b9753bfde551616224d0433ef71b2e614a8f6bf',
 'feature_rows': 5720,
 'cell_columns': 78262,
 'groups': 10,
 'single_gsm_per_condition': True,
 'common_expressed_features_for_rank_comparison': 3069,
 'analysis': 'Single-library pseudobulk CPM and gene-wise descriptive log2(CPM+1) ratios, separately by glucosamine stratum and time. No DE p-values or causal PfMYST interaction inference.'}

In [2]:
import pandas as pd
base = ROOT / 'research/transcriptomics/gse246115/results'
qc = pd.read_csv(base/'sample_qc.tsv', sep='\t')
contrasts = pd.read_csv(base/'contrast_summary.tsv', sep='\t')
rank_correlations = pd.read_csv(base/'response_rank_correlations.tsv', sep='\t', index_col=0)
print(qc.to_string(index=False))
print('\nDescriptive contrasts\n', contrasts.to_string(index=False))
print('\nGene-effect rank correlations\n', rank_correlations.to_string())

          group        gsm  cells  total_umis  median_umis_per_cell  median_detected_genes_per_cell
Myst_G0_DHA0_0h GSM7855693   4535     2680979                 547.0                           208.0
Myst_G0_DHA0_6h GSM7855694   6896     4165960                 603.0                           287.0
Myst_G0_DHA0_9h GSM7855695   8299     4670412                 555.0                           271.0
Myst_G0_DHA1_6h GSM7855696  11264     6180354                 530.0                           282.0
Myst_G0_DHA1_9h GSM7855697   9230     4743472                 488.0                           270.0
Myst_G1_DHA0_0h GSM7855698   4109     1106248                 261.0                           116.0
Myst_G1_DHA0_6h GSM7855699   5325     3153469                 589.0                           288.0
Myst_G1_DHA0_9h GSM7855700   5966     3506115                 575.0                           293.0
Myst_G1_DHA1_6h GSM7855701   9922     4921537                 499.0                           264.0


## Interpretation and limits

The four within-time DHA/control comparisons share gene-effect rank ordering in this processed matrix. This is **within-study descriptive agreement**, not independent reproduction: each group is one GEO sample/library, and cells within it are not biological replicates. Glucosamine changes PfMYST in a glmS line but also defines a treatment context; these aggregate contrasts do not isolate a causal PfMYST interaction. Parasite stage, cell-state composition, and sequencing depth can affect pseudobulk summaries. The source paper reports functional and multi-omics evidence separately; this notebook does not reproduce those experiments. A stronger follow-up would include independent culture replicates, explicit stage matching, count-based pseudobulk differential expression, and a genotype × DHA interaction model.